# 114 — LoRA Fine-Tuning

Attach low-rank adapters to a small language model, train on domain data, merge and export.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/114-lora-finetuning/lora_finetuning_workbook.ipynb)

In [ ]:
!pip install -q peft transformers datasets torch accelerate

## 1. What is LoRA?

LoRA (Low-Rank Adaptation, Hu et al. 2021) keeps the original model weights **frozen** and injects small trainable matrices into specific layers.

For a weight matrix W (shape d×k), LoRA adds:
```
W' = W + BA
```
Where:
- `B` is shape d×r (initialized to zero)
- `A` is shape r×k (initialized from random Gaussian)
- `r` (rank) << min(d, k) — typically 4–64

During training, only `A` and `B` are updated. After training, `W' = W + BA` can be merged back into W with zero inference overhead.

## 2. Why Low-Rank Adaptation Works

The weight updates during fine-tuning tend to have **low intrinsic rank** — the meaningful change in the weight matrix lives in a much smaller subspace than the full matrix dimensions.

LoRA exploits this:
- Full fine-tune: updates every parameter in W (d×k floats)
- LoRA r=8: only updates A (r×k) and B (d×r) — a tiny fraction

| Model Size | Full FT Params | LoRA r=8 Params | Savings |
|-----------|----------------|-----------------|--------|
| 135M | 135M | ~0.15M | ~99.9% |
| 7B | 7B | ~8M | ~99.9% |
| 70B | 70B | ~80M | ~99.9% |

## 3. PEFT LoraConfig Parameters

```python
LoraConfig(
    r=8,              # Rank — higher = more capacity, more params
    lora_alpha=16,    # Scaling factor (alpha/r is the effective scale)
    target_modules=["q_proj", "v_proj"],  # Which layers to adapt
    lora_dropout=0.05,   # Regularization dropout on LoRA layers
    bias="none",         # Don't train bias terms
    task_type=TaskType.CAUSAL_LM,
)
```

**Choosing target_modules**: attention projections (q, v) are most impactful for language tasks. You can also target `k_proj`, `o_proj`, or MLP layers for more capacity at the cost of more parameters.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(model_name)

total = sum(p.numel() for p in model.parameters())
print(f"Loaded: {model_name}")
print(f"Total parameters: {total/1e6:.1f}M")
print(f"All parameters trainable (full fine-tune would update all {total:,})")

## 4. Parameter Count — Before LoRA

With a standard model, every parameter is trainable. We'll count them now as a baseline to compare after attaching LoRA.

In [ ]:
def count_params(m):
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total = sum(p.numel() for p in m.parameters())
    return trainable, total

trainable_full, total_full = count_params(model)
print(f"Without LoRA:")
print(f"  Trainable: {trainable_full:,} ({100*trainable_full/total_full:.1f}% of total)")
print(f"  Total:     {total_full:,}")

## 5. Attaching LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model_lora = get_peft_model(model, lora_config)

trainable_lora, total_lora = count_params(model_lora)
print(f"With LoRA (r=8):")
print(f"  Trainable: {trainable_lora:,} ({100*trainable_lora/total_lora:.3f}% of total)")
print(f"  Total:     {total_lora:,}")
print(f"\nLoRA adds {trainable_lora - (total_full - trainable_full):,} new trainable parameters")
print(f"Parameter overhead: {100*trainable_lora/total_full:.2f}%")

## 6. Domain Dataset — Python Q&A

We use a built-in Python programming Q&A dataset. Format: instruction/response pairs with the template:

```
### Instruction:
{question}

### Response:
{answer}
```

This format teaches the model to follow instructions in our specific domain.

In [ ]:
import random

PYTHON_QA = [
    {"instruction": "How do you reverse a list in Python?", "output": "Use slicing: `my_list[::-1]` for a new list, or `my_list.reverse()` to reverse in place."},
    {"instruction": "What is a list comprehension in Python?", "output": "A concise syntax to build lists: `[x**2 for x in range(10) if x % 2 == 0]`."},
    {"instruction": "How do you handle exceptions in Python?", "output": "Use try/except: `try: risky_op() except ValueError as e: print(e) finally: cleanup()`."},
    {"instruction": "What is a lambda function in Python?", "output": "An anonymous function: `square = lambda x: x**2`. Useful with map/filter/sorted."},
    {"instruction": "How do you use enumerate() in Python?", "output": "`for i, val in enumerate(items):` — gives index and value. Start at 1 with `enumerate(items, start=1)`."},
    {"instruction": "What are *args and **kwargs?", "output": "`*args` collects positional args as a tuple; `**kwargs` collects keyword args as a dict."},
    {"instruction": "How do you open a file in Python?", "output": "`with open('file.txt', 'r') as f: content = f.read()` — context manager ensures the file closes."},
    {"instruction": "What is a dictionary in Python?", "output": "A key-value mapping: `d = {'a': 1}`. Access with `d['a']`, iterate with `d.items()`."},
    {"instruction": "What is the difference between `is` and `==`?", "output": "`==` compares values; `is` compares identity (same object in memory)."},
    {"instruction": "How do you sort a list of dicts by a key?", "output": "`sorted(people, key=lambda x: x['age'])` — returns sorted list."},
]

random.seed(42)
dataset_raw = random.choices(PYTHON_QA, k=50)
print(f"Dataset: {len(dataset_raw)} instruction/response pairs")
print(f"\nSample:")
print(f"  Instruction: {dataset_raw[0]['instruction']}")
print(f"  Output:      {dataset_raw[0]['output'][:80]}...")

In [ ]:
from datasets import Dataset

def format_example(ex):
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

texts = [format_example(ex) for ex in dataset_raw]

def tokenize_fn(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

raw_ds = Dataset.from_dict({"text": texts})
train_dataset = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f"Tokenized dataset: {len(train_dataset)} examples")
print(f"Features: {list(train_dataset.features.keys())}")
print(f"Sequence length: {len(train_dataset[0]['input_ids'])} tokens (max_length=256)")

## 7. Before Training — Base Model Responses

In [ ]:
import torch

def generate(m, tok, prompt, max_new_tokens=100):
    full = f"### Instruction:\n{prompt}\n\n### Response:\n"
    inputs = tok(full, return_tensors="pt")
    with torch.no_grad():
        out = m.generate(
            inputs["input_ids"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

test_prompts = [
    "How do you reverse a list in Python?",
    "What is a list comprehension in Python?",
    "How do you handle exceptions in Python?",
]

print("=== BASE MODEL (before LoRA training) ===\n")
responses_before = []
for p in test_prompts:
    r = generate(model_lora, tokenizer, p)
    responses_before.append(r)
    print(f"Q: {p}")
    print(f"A: {r[:150]}\n")

## 8. LoRA Training

We use HuggingFace `Trainer` with standard `TrainingArguments`. Key settings:
- `learning_rate=2e-4` — slightly higher than full fine-tuning (LoRA adapters start at zero)
- `per_device_train_batch_size=2` — small batch works on CPU for 135M model

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./lora_out",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Starting LoRA training...")
result = trainer.train()
print(f"\nTraining complete! Loss: {result.training_loss:.4f}")

## 9. Merging Adapters into Base Weights

After training, `merge_and_unload()` computes `W' = W + BA` for every adapted layer and stores the result in the weight matrix directly. The adapter is then removed — the model looks identical to the original but with updated weights.

**Zero inference overhead**: merged weights are standard tensors, no PEFT wrapper needed at deployment.

In [ ]:
print("Merging LoRA adapters...")
merged_model = model_lora.merge_and_unload()

trainable_merged, total_merged = count_params(merged_model)
print(f"After merge:")
print(f"  Trainable: {trainable_merged:,} (all params, no adapter wrapper)")
print(f"  Total:     {total_merged:,}")
print(f"\nModel is now a standard CausalLM — no PEFT overhead at inference.")

## 10. After Training — Fine-Tuned Responses

In [ ]:
print("=== FINE-TUNED MODEL (after LoRA training + merge) ===\n")
responses_after = []
for p in test_prompts:
    r = generate(merged_model, tokenizer, p)
    responses_after.append(r)
    print(f"Q: {p}")
    print(f"A: {r[:150]}\n")

## 11. Side-by-Side Comparison

In [ ]:
print(f"{'Prompt':<45} {'Before (chars)':<20} {'After (chars)':<20}")
print("-" * 86)
for p, b, a in zip(test_prompts, responses_before, responses_after):
    print(f"{p[:44]:<45} {len(b):<20} {len(a):<20}")

print("\nDetailed comparison:")
for p, b, a in zip(test_prompts, responses_before, responses_after):
    print(f"\nQ: {p}")
    print(f"  Before: {b[:120]}")
    print(f"  After:  {a[:120]}")

## Exercises

### Exercise 1 — Increase Rank

Change `r=8` to `r=16` and retrain. How many additional parameters does this add? Use `count_params()` to compare. Does higher rank improve response quality?

In [ ]:
# Exercise 1: Try r=16 and compare parameter counts
from peft import LoraConfig, get_peft_model, TaskType

lora_config_r16 = LoraConfig(
    r=16,              # <-- changed from 8
    lora_alpha=32,     # keep alpha/r ratio = 2
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
# model_r16 = get_peft_model(model, lora_config_r16)
# trainable_r16, total_r16 = count_params(model_r16)
# print(f"r=16: {trainable_r16:,} trainable ({100*trainable_r16/total_r16:.3f}%)")
print("Exercise 1: Attach LoraConfig with r=16 and compare param counts to r=8.")

### Exercise 2 — Add More Target Modules

Expand `target_modules` to include `["q_proj", "k_proj", "v_proj", "o_proj"]`. How does this change the trainable parameter count and the training time?

### Exercise 3 — Save and Load the Adapter

After training (before merging), save the LoRA adapter weights only with `model_lora.save_pretrained('./my_adapter')`. Load it back with `PeftModel.from_pretrained(base_model, './my_adapter')`. Verify the loaded model produces the same outputs.

In [ ]:
# Answer Key

# Exercise 1 — r=16 doubles the adapter parameters vs r=8
# r=8:  A is (r×k) = (8×64), B is (d×r) = (64×8) per layer -> 2*(8*64 + 64*8) per head
# r=16: A is (16×64), B is (64×16) per layer -> exactly 2x the adapter params
# Higher rank gives more capacity but risks overfitting on small datasets.

# Exercise 2 — 4 modules vs 2 modules
# Adding k_proj and o_proj roughly doubles the trainable params again.
# Training time increases proportionally; may help on tasks requiring richer attention.

# Exercise 3 — Adapter save/load
from peft import PeftModel
# Save: model_lora.save_pretrained('./my_adapter')
# Load: loaded = PeftModel.from_pretrained(base_model, './my_adapter')
# Verify: generate(loaded, tokenizer, test_prompts[0]) should match pre-merge output.
print("Answer key reviewed. See comments above for explanations.")